# ComfyUI workspace · H3 Continuum / Ref2VA-VSA / FastH3 Omni

獨立的 ComfyUI 操作入口：內嵌工具，不需 clone 本專案。預設勾選安裝 Continuum 與 Ref2VA-VSA 擴充；A 區會安裝並啟動 server；模型預設不下載，B 區預設不生成。先跑 1–2 檢視版本與磁碟下載 GiB，再執行 3–5；要下載時明確選 MODEL_BUNDLE 並重跑設定、計畫與下載格。

這裡沒有 GPU 實測或效能保證。請自行選擇合適 Colab GPU；模型下載量不等於 VRAM。Ref2VA-VSA 是實驗性原生 Ref2VA，需專用 Ref2VA 模型、Turbo 與 gate，不沿用 FastH3 T2VA student。兩者是分開的 starter，沒有宣稱可組合。Veda 本次暫不實作，僅保留[研究](../research/comfy_h3_extensions_20261019.md)。

新增 `h3-omni-int8`：[FastVideo FastH3 Omni](https://huggingface.co/FastVideo/FastH3-Omni-Comfy) 的原生 Ref2VA INT8 模型，使用 8 步、guidance 1、shift 12/3，不疊 Turbo LoRA 或 VSA gate transplant。搭配含 vision tower 的 INT8 Qwen3-VL encoder、INT8 video VAE 與 FP32 audio VAE；不需要額外 custom node。模型 bundle 約 49.15 GiB，並非 VRAM 需求或容量保證。

使用獨立 root `/content/comfyui-workspace`、port `8190`，與 FastH3 的 root/8188 分開。安裝使用隔離 venv 並保留 Colab 現有 Torch/CUDA。模型受各自授權條款限制。詳細步驟見[操作說明](../doc/COMFYUI_WORKSPACE.md)。

## A. 預覽、安裝與開啟 UI


In [ ]:
# @title 1. 設定與還原工具（不安裝、不下載模型）
import json
import sys
from pathlib import Path
WORKSPACE_ROOT = "/content/comfyui-workspace" # @param {type:"string"}
COMFYUI_PORT = 8190 # @param {type:"integer"}
TOOLKIT_ROOT = "/content/comfyui-workspace-toolkit" # @param {type:"string"}
ENABLE_CONTINUUM = True # @param {type:"boolean"}
ENABLE_REF2VA_VSA = True # @param {type:"boolean"}
MODEL_BUNDLE = "none" # @param ["none", "h3-continuum", "h3-ref2va-vsa", "h3-omni-int8"]
INSTALL_COMFYUI = True # @param {type:"boolean"}
START_COMFYUI = True # @param {type:"boolean"}
SHOW_COMFYUI = True # @param {type:"boolean"}
# BEGIN EMBEDDED COMFYUI WORKSPACE TOOLKIT
COMFYUI_SOURCES = {'comfyui_workspace.py': '#!/usr/bin/env python3\n"""Pinned, opt-in ComfyUI workspace for Colab; imports only the standard library.\n\nThis is independent of the verified L4 FastH3 installation. Installing nodes or\nstarting the server does not establish that a model fits or produces good media.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom urllib.parse import quote\nfrom urllib.request import urlopen\nimport uuid\nimport shutil\n\nfrom fast_h3_client import ComfyClient, PromptExecutionError, PromptTimeoutError\n\n\nCATALOG_PATH = Path(__file__).with_name("comfyui_catalog.json")\nDEFAULT_EXTENSIONS = ("continuum", "ref2va-vsa")\n\n\ndef load_catalog():\n    return json.loads(CATALOG_PATH.read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef model_path(root, destination):\n    relative = PurePosixPath(destination)\n    if (relative.is_absolute() or not relative.parts or ".." in relative.parts\n            or "\\\\" in destination):\n        raise ValueError("Model destination must be relative to ComfyUI/models")\n    root = Path(root).resolve()\n    target = (root / str(relative)).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError("Model destination leaves ComfyUI/models")\n    return target\n\n\ndef download_asset(asset, target):\n    """Stream one pinned artifact, verifying it before atomic publication.\n\n    Existing different files are never overwritten. A retry can reuse completed\n    files; an incomplete temporary transfer is discarded on failure.\n    """\n    if not re.fullmatch(r"[0-9a-f]{40}", asset["revision"]):\n        raise ValueError("Model revision must be a full commit SHA")\n    if not re.fullmatch(r"[0-9a-f]{64}", asset["sha256"]):\n        raise ValueError("Model must have a SHA-256")\n    target = Path(target)\n    if target.exists():\n        if target.stat().st_size != asset["size"] or sha256(target) != asset["sha256"]:\n            raise RuntimeError(f"Existing model differs from the catalog: {target}")\n        return target\n    target.parent.mkdir(parents=True, exist_ok=True)\n    url = ("https://huggingface.co/" + quote(asset["repo"], safe="/") + "/resolve/"\n           + asset["revision"] + "/" + quote(asset["file"], safe="/"))\n    descriptor, temporary = tempfile.mkstemp(prefix="." + target.name + ".", suffix=".part", dir=target.parent)\n    digest, received = hashlib.sha256(), 0\n    try:\n        with os.fdopen(descriptor, "wb") as output, urlopen(url, timeout=120) as response:\n            for chunk in iter(lambda: response.read(8 * 1024 * 1024), b""):\n                received += len(chunk)\n                if received > asset["size"]:\n                    raise RuntimeError(f"Model exceeds its declared size: {target.name}")\n                output.write(chunk)\n                digest.update(chunk)\n            output.flush()\n            os.fsync(output.fileno())\n        if received != asset["size"] or digest.hexdigest() != asset["sha256"]:\n            raise RuntimeError(f"Model size/SHA-256 mismatch: {target.name}")\n        os.link(temporary, target)\n        return target\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\nclass ComfyWorkspace:\n    def __init__(self, root="/content/comfyui-workspace", port=8190, extensions=DEFAULT_EXTENSIONS):\n        self.root = Path(root).expanduser().resolve()\n        self.comfy = self.root / "ComfyUI"\n        self.state = self.root / "state"\n        self.python = self.root / "venv/bin/python"\n        if type(port) is not int or not 1024 <= port <= 65535:\n            raise ValueError("Use a port between 1024 and 65535")\n        self.port = port\n        self.catalog = load_catalog()\n        if isinstance(extensions, str):\n            extensions = [name.strip() for name in extensions.split(",") if name.strip()]\n        unknown = set(extensions) - self.catalog["extensions"].keys()\n        if unknown:\n            raise ValueError(f"Unknown extensions: {sorted(unknown)}")\n        self.extensions = tuple(sorted(set(extensions)))\n        self.client = ComfyClient(f"http://127.0.0.1:{port}")\n        self.server = None\n\n    def _signature(self):\n        selected = {key: self.catalog["extensions"][key] for key in self.extensions}\n        return hashlib.sha256(json.dumps({"comfyui": self.catalog["comfyui"],\n                                         "extensions": selected}, sort_keys=True).encode()).hexdigest()\n\n    def _bundle(self, name):\n        if name not in self.catalog["bundles"]:\n            raise ValueError(f"Unknown model bundle: {name}")\n        bundle = self.catalog["bundles"][name]\n        missing = set(bundle["extensions"]) - set(self.extensions)\n        if missing:\n            raise ValueError(f"Enable extensions for {name}: {sorted(missing)}")\n        return bundle\n\n    def plan(self, bundle="none"):\n        selected = self._bundle(bundle)\n        assets = [self.catalog["assets"][key] for key in selected["assets"]]\n        return {"root": str(self.root), "port": self.port, "comfyui": self.catalog["comfyui"],\n                "extensions": {key: self.catalog["extensions"][key] for key in self.extensions},\n                "bundle": bundle, "description": selected["description"], "assets": assets,\n                "download_bytes": sum(item["size"] for item in assets),\n                "download_gib": round(sum(item["size"] for item in assets) / 2**30, 2),\n                "gpu_generation": "not_run"}\n\n    def _run(self, command, cwd=None):\n        self.state.mkdir(parents=True, exist_ok=True)\n        print("ComfyUI:", " ".join(map(str, command)), flush=True)\n        with (self.state / "install.log").open("a") as log:\n            result = subprocess.run(list(map(str, command)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)\n        if result.returncode:\n            raise RuntimeError(f"Command exited {result.returncode}; see {self.state / \'install.log\'}")\n\n    def _checkout(self, spec, destination):\n        if (destination / ".git").is_dir():\n            actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()\n            if actual != spec["revision"]:\n                raise RuntimeError(f"Existing checkout has another revision: {destination}; use a new workspace root")\n            return\n        if destination.exists() and any(destination.iterdir()):\n            raise RuntimeError(f"Refusing to replace a nonempty directory: {destination}")\n        destination.mkdir(parents=True, exist_ok=True)\n        self._run(["git", "init", destination])\n        self._run(["git", "-C", destination, "remote", "add", "origin", spec["repo"]])\n        self._run(["git", "-C", destination, "fetch", "--depth", "1", "origin", spec["revision"]])\n        self._run(["git", "-C", destination, "checkout", "--detach", "FETCH_HEAD"])\n\n    def _versions(self):\n        code = """import importlib.metadata as m,json,torch\nnames=(\'torch\',\'torchvision\',\'torchaudio\',\'triton\',\'comfy-kitchen\',\'safetensors\')\nversions={}\nfor name in names:\n    try: versions[name]=m.version(name)\n    except m.PackageNotFoundError: pass\nprint(json.dumps({\'packages\':versions,\'cuda\':torch.version.cuda,\'cuda_available\':torch.cuda.is_available(),\n \'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))\n"""\n        return json.loads(subprocess.check_output([str(self.python), "-c", code], text=True).strip().splitlines()[-1])\n\n    def install(self):\n        if self._port_open():\n            self._require_owned_server()\n            marker = self._server_marker()\n            if marker["signature"] != self._signature():\n                raise RuntimeError("Stop this workspace\'s server before changing enabled extensions")\n            return self.status()\n        old = self._server_marker()\n        identity = self._process_identity(old.get("pid")) if old else None\n        if identity and identity == old.get("identity"):\n            raise RuntimeError("Stop the existing workspace server before installing, including a server on another port")\n        self.state.mkdir(parents=True, exist_ok=True)\n        self._checkout(self.catalog["comfyui"], self.comfy)\n        for name in self.extensions:\n            spec = self.catalog["extensions"][name]\n            self._checkout(spec, self.comfy / "custom_nodes" / spec["directory"])\n        installed = self.state / "installed.json"\n        if installed.exists() and json.loads(installed.read_text()).get("signature") == self._signature() and self.python.is_file():\n            return json.loads(installed.read_text())\n        # Colab exposes pip but can disable ensurepip. Keep its working GPU stack.\n        self._run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", self.root / "venv"])\n        preserved = {}\n        for distribution in importlib.metadata.distributions():\n            name = distribution.metadata.get("Name", "").lower().replace("_", "-")\n            if name in ("torch", "torchvision", "torchaudio", "triton") or name.startswith("nvidia-"):\n                preserved[name] = distribution.version\n        if "torch" not in preserved:\n            raise RuntimeError("A preinstalled PyTorch/CUDA stack is required; use a standard Colab GPU runtime")\n        constraints = [f"{name}=={version}" for name, version in sorted(preserved.items())]\n        constraint_file = self.state / "gpu-constraints.txt"\n        constraint_file.write_text("\\n".join(constraints) + "\\n")\n        requirements = [item for name in self.extensions for item in self.catalog["extensions"][name]["requirements"]]\n        self._run([self.python, "-m", "pip", "install", "--disable-pip-version-check", "-c", constraint_file,\n                   "-r", self.comfy / "requirements.txt", *requirements])\n        versions = self._versions()\n        if any(versions["packages"].get(name) != version for name, version in preserved.items()\n               if name in versions["packages"]):\n            raise RuntimeError("The effective GPU stack changed; restart with a compatible runtime")\n        freeze = subprocess.check_output([str(self.python), "-m", "pip", "freeze"], text=True)\n        (self.state / "versions.txt").write_text(freeze)\n        result = {"signature": self._signature(), "plan": self.plan(), "environment": versions,\n                  "gpu_generation": "not_run"}\n        write_json(installed, result)\n        return result\n\n    def _preflight_bundle(self, bundle):\n        code = """import json,torch\nassert torch.cuda.is_available(), \'Model bundles require a CUDA GPU\'\np=torch.cuda.get_device_properties(0)\nresult={\'gpu\':p.name,\'vram_bytes\':p.total_memory,\'capability\':list(torch.cuda.get_device_capability())}\n"""\n        if bundle == "h3-ref2va-vsa":\n            code += """assert p.major >= 8 and p.total_memory >= 22*2**30, \'Ref2VA-VSA needs a 24 GB class NVIDIA Ampere+ GPU\'\nimport comfy_kitchen as ck\nfrom comfy_kitchen.backends import cuda\nassert cuda._EXT_AVAILABLE, str(cuda._EXT_ERROR)\nq=torch.randn(1,256,4,128,device=\'cuda\',dtype=torch.bfloat16)\nwith ck.use_backend(\'cuda\'):\n    out=ck.sol_attn(q,torch.randn_like(q),torch.randn_like(q),topk_ratio=0.25,tail=False,\n      sink_blocks=[0,1],sink_q=[0,1],block_len=torch.full((4,),64,device=\'cuda\',dtype=torch.int32),\n      coarse_gate=torch.zeros_like(q))\nassert out.shape==q.shape and torch.isfinite(out).all(), \'Sol CUDA kernel probe failed\'\ntorch.cuda.synchronize()\nresult[\'sol_kernel\']=\'pass\'\n"""\n        code += "print(json.dumps(result))\\n"\n        self.state.mkdir(parents=True, exist_ok=True)\n        probe = subprocess.run([str(self.python), "-c", code], capture_output=True, text=True)\n        (self.state / "model-preflight.log").write_text(probe.stdout + probe.stderr)\n        if probe.returncode:\n            raise RuntimeError(f"Model hardware/kernel preflight failed before downloads; see {self.state / \'model-preflight.log\'}")\n        result = json.loads(probe.stdout.strip().splitlines()[-1])\n        write_json(self.state / "model-preflight.json", {"bundle": bundle, **result})\n        return result\n\n    def download_models(self, bundle="none"):\n        selected = self._bundle(bundle)\n        if not selected["assets"]:\n            return {"bundle": bundle, "downloaded": [], "status": "no_models_requested"}\n        if self._port_open():\n            raise RuntimeError("Stop the ComfyUI server before downloading models, then start it to refresh model lists")\n        installed = self.state / "installed.json"\n        if (not installed.is_file() or not self.python.is_file()\n                or json.loads(installed.read_text()).get("signature") != self._signature()):\n            raise RuntimeError("Install the workspace before downloading models")\n        self._preflight_bundle(bundle)\n        assets = [self.catalog["assets"][key] for key in selected["assets"]]\n        missing_bytes = sum(item["size"] for item in assets if not model_path(self.comfy / "models", item["destination"]).exists())\n        if shutil.disk_usage(self.root).free < missing_bytes + 2 * 2**30:\n            raise RuntimeError("Insufficient disk for the selected weights plus 2 GiB headroom")\n        paths = []\n        for item in assets:\n            print("Model:", item["destination"], flush=True)\n            path = download_asset(item, model_path(self.comfy / "models", item["destination"]))\n            paths.append(str(path))\n            write_json(self.state / "download-progress.json", {"bundle": bundle, "verified_paths": paths})\n        result = {"bundle": bundle, "status": "verified", "paths": paths,\n                  "assets": assets, "gpu_generation": "not_run"}\n        write_json(self.state / (bundle + "-models.json"), result)\n        return result\n\n    def stage_workflows(self):\n        from comfyui_workflows import export\n        return [str(path) for path in export(self.root / "workflows", presets=[\n            key for key, spec in self.catalog["bundles"].items()\n            if key != "none" and set(spec["extensions"]).issubset(self.extensions)])]\n\n    def _port_open(self):\n        try:\n            with socket.create_connection(("127.0.0.1", self.port), timeout=1):\n                return True\n        except OSError:\n            return False\n\n    def _server_marker(self):\n        marker = self.state / "server.json"\n        return json.loads(marker.read_text()) if marker.exists() else {}\n\n    @staticmethod\n    def _process_identity(pid):\n        # Linux /proc start time prevents a stale PID from owning another server.\n        try:\n            process = Path("/proc") / str(int(pid))\n            stat = (process / "stat").read_text().rsplit(")", 1)[1].split()\n            if stat[0] == "Z":\n                return None\n            return {"start_ticks": stat[19], "command": (process / "cmdline").read_bytes().decode().rstrip("\\0").split("\\0")}\n        except (OSError, ValueError, TypeError):\n            return None\n\n    def _require_owned_server(self):\n        marker = self._server_marker()\n        identity = self._process_identity(marker.get("pid"))\n        if (not identity or identity != marker.get("identity") or marker.get("port") != self.port\n                or marker.get("command") != identity["command"]\n                or str(self.comfy / "main.py") not in identity["command"]):\n            raise RuntimeError("The port/process is not owned by this workspace; use another port/root")\n        return marker\n\n    def start(self, timeout=180):\n        if timeout <= 0:\n            raise ValueError("Startup timeout must be positive")\n        if self._port_open():\n            marker = self._require_owned_server()\n            if marker["signature"] != self._signature():\n                raise RuntimeError("Stop the server before changing enabled extensions")\n            return self._ready()\n        old = self._server_marker()\n        identity = self._process_identity(old.get("pid")) if old else None\n        if identity and identity == old.get("identity"):\n            raise RuntimeError("The owned server is still starting or unhealthy; inspect server.log before restarting")\n        installed = self.state / "installed.json"\n        if not installed.exists() or json.loads(installed.read_text()).get("signature") != self._signature():\n            raise RuntimeError("Install the selected workspace/extensions before starting")\n        directories = [self.catalog["extensions"][name]["directory"] for name in self.extensions]\n        command = [str(self.python), str(self.comfy / "main.py"), "--listen", "127.0.0.1", "--port", str(self.port),\n                   "--disable-auto-launch", "--disable-comfy-compiler", "--use-pytorch-cross-attention",\n                   "--disable-all-custom-nodes"]\n        if directories:\n            command += ["--whitelist-custom-nodes", *directories]\n        with (self.state / "server.log").open("a") as log:\n            self.server = subprocess.Popen(command, cwd=self.comfy, stdout=log, stderr=subprocess.STDOUT,\n                                           start_new_session=True)\n        identity = self._process_identity(self.server.pid)\n        if not identity:\n            raise RuntimeError("Cannot verify the ComfyUI process; see server.log")\n        write_json(self.state / "server.json", {"pid": self.server.pid, "port": self.port,\n                    "command": command, "identity": identity, "signature": self._signature()})\n        deadline = time.monotonic() + timeout\n        while time.monotonic() < deadline:\n            if self.server.poll() is not None:\n                raise RuntimeError("ComfyUI startup failed; see server.log")\n            if self._port_open():\n                try:\n                    return self._ready()\n                except OSError:\n                    pass\n            time.sleep(1)\n        raise TimeoutError("ComfyUI is still starting; inspect status/server.log before retrying")\n\n    def _ready(self):\n        marker = self._require_owned_server()\n        if marker["signature"] != self._signature():\n            raise RuntimeError("Server extension selection differs; stop, install the selection and restart")\n        schemas = self.client.object_info()\n        expected = {node for name in self.extensions for node in self.catalog["extensions"][name]["expected_nodes"]}\n        missing = expected - schemas.keys()\n        if missing:\n            raise RuntimeError(f"Custom nodes did not load: {sorted(missing)}; inspect server.log")\n        write_json(self.state / "object-info.json", schemas)\n        return {"server": "ready", "base_url": self.client.base_url, "extensions": list(self.extensions),\n                "node_count": len(schemas), "gpu_generation": "not_run"}\n\n    def status(self):\n        if not self._port_open():\n            marker = self._server_marker()\n            identity = self._process_identity(marker.get("pid")) if marker else None\n            alive = bool(identity and identity == marker.get("identity"))\n            return {"server": "starting_or_unhealthy" if alive else "stopped", "root": str(self.root)}\n        result = self._ready()\n        result["queue"] = self.client.get_json("/queue")\n        return result\n\n    def stop(self):\n        marker = self._server_marker()\n        if not marker or self._process_identity(marker.get("pid")) is None:\n            return {"server": "stopped", "colab_runtime": "unchanged"}\n        marker = self._require_owned_server()\n        os.kill(marker["pid"], signal.SIGTERM)\n        if self.server is not None:\n            self.server.wait(timeout=30)\n        deadline = time.monotonic() + 30\n        while self._process_identity(marker["pid"]) == marker["identity"]:\n            if time.monotonic() >= deadline:\n                raise TimeoutError("ComfyUI has not stopped; inspect status before downloading or restarting")\n            time.sleep(0.25)\n        return {"server": "stopped", "colab_runtime": "unchanged"}\n\n    def run_api(self, workflow_path, timeout=1800):\n        if timeout <= 0:\n            raise ValueError("Execution timeout must be positive")\n        graph = json.loads(Path(workflow_path).read_text())\n        if not isinstance(graph, dict) or "nodes" in graph or not graph:\n            raise ValueError("Use a ComfyUI API graph, not a UI workflow export")\n        from comfyui_workflows import validate_attention\n        validate_attention(graph)\n        marker = self._require_owned_server()\n        if marker["signature"] != self._signature():\n            raise RuntimeError("Server extension selection differs; stop, install the selection and restart")\n        self.client.validate_graph(graph)\n        identity = str(uuid.uuid4())\n        run_dir = self.root / "runs" / identity\n        run_dir.mkdir(parents=True)\n        write_json(run_dir / "workflow.api.json", graph)\n        receipt = {"prompt_id": identity, "status": "submitting", "run_dir": str(run_dir),\n                   "catalog": self.catalog, "media_validation": "not_run", "content_review": "not_run"}\n        write_json(run_dir / "receipt.json", receipt)\n        try:\n            response = self.client.queue_prompt(graph, prompt_id=identity)\n            if response["prompt_id"] != identity:\n                raise RuntimeError("ComfyUI returned a different prompt identity; inspect queue/history")\n            receipt["status"] = "queued"\n            write_json(run_dir / "receipt.json", receipt)\n            history = self.client.wait_prompt(identity, timeout=timeout)\n            write_json(run_dir / "history.json", history)\n            receipt.update(status="execution_complete", outputs=history.get("outputs", {}))\n        except PromptExecutionError as error:\n            write_json(run_dir / "history.json", error.history)\n            receipt.update(status="execution_failed", error_type=type(error).__name__)\n            raise\n        except PromptTimeoutError:\n            receipt["status"] = "timeout_may_be_running"\n            raise\n        except Exception as error:\n            receipt.update(status="submission_or_connection_unknown", error_type=type(error).__name__)\n            raise\n        finally:\n            write_json(run_dir / "receipt.json", receipt)\n            print("Receipt:", run_dir / "receipt.json", "Prompt:", identity, flush=True)\n        return receipt\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=("plan", "install", "download", "start", "status", "stop", "run", "workflows"))\n    parser.add_argument("--root", type=Path, default=Path("/content/comfyui-workspace"))\n    parser.add_argument("--port", type=int, default=8190)\n    parser.add_argument("--extensions", default=",".join(DEFAULT_EXTENSIONS))\n    parser.add_argument("--bundle", default="none", choices=tuple(load_catalog()["bundles"]))\n    parser.add_argument("--workflow", type=Path)\n    parser.add_argument("--timeout", type=int, default=1800)\n    args = parser.parse_args()\n    workspace = ComfyWorkspace(args.root, args.port, args.extensions)\n    if args.command == "run" and args.workflow is None:\n        parser.error("run requires --workflow")\n    actions = {"plan": lambda: workspace.plan(args.bundle), "install": workspace.install,\n               "download": lambda: workspace.download_models(args.bundle), "start": workspace.start,\n               "status": workspace.status, "stop": workspace.stop,\n               "run": lambda: workspace.run_api(args.workflow, args.timeout), "workflows": workspace.stage_workflows}\n    print(json.dumps(actions[args.command](), indent=2, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'comfyui_workflows.py': '#!/usr/bin/env python3\n"""Small native H3 starters for the optional ComfyUI workspace (no GPU imports)."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport uuid\n\n\nPRESETS = ("h3-continuum", "h3-ref2va-vsa", "h3-omni-int8")\nDEFAULT_PROMPT = (\n    "integrated_multimodal_description: [Shot 1] A small red toy train rolls "\n    "steadily across a wooden table. Static wide camera, soft daylight.\\n"\n    "overall_soundscape: Wheels rattle softly against the wood.\\n"\n    "non_diegetic_music: N/A."\n)\nREFERENCE_PROMPT = (\n    "subject_definitions: <Subject 1> is the red toy train in <Picture 1>.\\n"\n    "summary: [reference generation] The referenced toy train rolls across a table.\\n"\n    "retention_analysis: <Subject 1> (appears in [Shot 1]): fully_preserved.\\n"\n    "detailed_description: [Shot 1] <Subject 1> from <Picture 1> rolls steadily "\n    "across a wooden table. Static wide camera, soft daylight.\\n"\n    "overall_soundscape: Wheels rattle softly against the wood.\\n"\n    "non_diegetic_music: N/A."\n)\n\n\ndef node(class_type, **inputs):\n    return {"class_type": class_type, "inputs": inputs}\n\n\ndef validate_attention(graph):\n    """Reject a second attention patch along a Ref2VA-VSA model wire.\n\n    Other custom patch conflicts remain the node\'s responsibility at execution;\n    importing an unrelated branch\'s attention patch is not itself a conflict.\n    """\n    for identity, item in graph.items():\n        if not isinstance(item, dict):\n            raise ValueError("API graph values must be node dictionaries")\n        chain, visited, current = [], set(), identity\n        while current in graph and current not in visited:\n            visited.add(current)\n            entry = graph[current]\n            kind = entry.get("class_type", "")\n            chain.append(kind)\n            model = entry.get("inputs", {}).get("model")\n            if not (isinstance(model, list) and len(model) == 2 and isinstance(model[0], str)):\n                break\n            current = model[0]\n        if "Ref2VAVSAGatePatch" in chain:\n            patches = [kind for kind in chain if kind == "Ref2VAVSAGatePatch"\n                       or any(part in kind.lower() for part in ("attention", "vsapatch", "solattn"))]\n            if len(patches) > 1:\n                raise ValueError("Ref2VA-VSA cannot be stacked with another attention patch on the same model")\n    return graph\n\n\ndef build_graph(preset, prompt=None, width=864, height=480, duration_seconds=5,\n                seed=42, reference_image="reference.png", chunks=2, run_name="comfy_h3",\n                ref_image_size=None):\n    if preset not in PRESETS:\n        raise ValueError(f"Unknown preset: {preset}")\n    if any(type(value) is not int or value < 32 or value % 32 for value in (width, height)):\n        raise ValueError("Dimensions must be positive multiples of 32")\n    if width * height > 1344 * 768:\n        raise ValueError("Starter canvas must not exceed 1344x768; use a custom API graph for other sizes")\n    if type(seed) is not int or not 0 <= seed < 2**64:\n        raise ValueError("Seed must be a uint64 integer")\n    if type(duration_seconds) not in (int, float) or not math.isfinite(duration_seconds):\n        raise ValueError("Duration must be finite")\n    if type(chunks) is not int or not 1 <= chunks <= 16:\n        raise ValueError("Chunks must be from 1 to 16")\n    if not isinstance(run_name, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", run_name):\n        raise ValueError("Run name must be filename-safe")\n    omni = preset == "h3-omni-int8"\n    reference = preset in ("h3-ref2va-vsa", "h3-omni-int8")\n    # Preserve the existing Ref2VA starter\'s sizing; Omni favors reference detail.\n    if ref_image_size is None:\n        ref_image_size = "max" if omni else "match"\n    if ref_image_size not in ("match", "max"):\n        raise ValueError("Reference image size must be match or max")\n    if prompt is None or prompt == "":\n        prompt = REFERENCE_PROMPT if reference else DEFAULT_PROMPT\n    if not isinstance(prompt, str) or not prompt.strip():\n        raise ValueError("Prompt must be nonempty text")\n    if not (5 <= duration_seconds <= 15 if reference else 4 <= duration_seconds <= 30):\n        raise ValueError("Duration must be 5–15 seconds for Ref2VA, or 4–30 seconds per Continuum chunk")\n    if reference:\n        if (not isinstance(reference_image, str) or not reference_image or "\\\\" in reference_image\n                or PurePosixPath(reference_image).is_absolute() or ".." in PurePosixPath(reference_image).parts\n                or PurePosixPath(reference_image).suffix.lower() not in (".png", ".jpg", ".jpeg", ".webp")):\n            raise ValueError("Reference must be a relative ComfyUI input image filename")\n        fields = ("subject_definitions", "summary", "retention_analysis", "detailed_description",\n                  "overall_soundscape", "non_diegetic_music")\n        pattern = r"\\A\\s*" + r"\\s*".join(field + r":\\s*(.+?)" for field in fields) + r"\\s*\\Z"\n        match = re.fullmatch(pattern, prompt, flags=re.DOTALL)\n        if not match or not all(value.strip() for value in match.groups()):\n            raise ValueError("Ref2VA prompt requires all six nonempty fields in canonical order")\n        if not match.group(2).lstrip().startswith("[reference generation]"):\n            raise ValueError("Ref2VA summary must start with [reference generation]")\n        tags = re.findall(r"<(Picture|Video|Audio)\\s+(\\d+)>", prompt, flags=re.IGNORECASE)\n        if {(kind.lower(), int(number)) for kind, number in tags} != {("picture", 1)}:\n            raise ValueError("This starter requires exactly the <Picture 1> reference")\n\n    graph = {\n        "1": node("UNETLoader", unet_name=("fasth3_omniref_8step_pruned_int8_convrot.safetensors" if omni\n                                         else "minimax_h3_ref2va_pruned_int8_convrot.safetensors" if reference\n                                         else "minimax_h3_fl2va_pruned_int8_convrot.safetensors"), weight_dtype="default"),\n        "2": node("CLIPLoader", clip_name=("qwen3vl_32b_minimax_h3_int8_convrot.safetensors" if omni\n                                         else "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"), type="minimax", device="default"),\n        "3": node("VAELoader", vae_name=("minimax_h3_video_vae_fp16.safetensors" if reference and not omni\n                                       else "minimax_h3_video_vae_int8_convrot.safetensors")),\n        "4": node("VAELoader", vae_name="minimax_h3_audio_vae_fp32.safetensors"),\n        "5": node("KSamplerSelect", sampler_name="euler" if reference else "res_multistep"),\n    }\n    if reference:\n        if not omni:\n            graph.update({\n                "6": node("LoraLoaderModelOnly", model=["1", 0],\n                          lora_name="minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", strength_model=1.0),\n                "7": node("Ref2VAVSAGatePatch", model=["6", 0], gate_file="fasth3_vsa_gate.safetensors", sparsity=0.75),\n            })\n        graph.update({\n            # Omni already contains its eight-step distillation and VSA gate weights.\n            # Use native dense attention here; no Turbo LoRA or gate transplant.\n            "8": node("MiniMaxH3SigmaShift", model=["1" if omni else "7", 0], shift_video=12.0, shift_audio=3.0),\n            "9": node("BasicScheduler", model=["8", 0], scheduler="simple", steps=8 if omni else 4, denoise=1.0),\n            "10": node("LoadImage", image=reference_image),\n            "11": node("MiniMaxH3ReferenceToVideo", clip=["2", 0], vae=["3", 0], audio_vae=["4", 0],\n                       prompt=prompt, width=width, height=height,\n                       length=(max(5, round(duration_seconds * 24)) - 5 + 16) // 17 * 17 + 5,\n                       ref_image_size=ref_image_size, **{"ref_images.ref_image_0": ["10", 0]}),\n            "12": node("RandomNoise", noise_seed=seed),\n            "13": node("BasicGuider", model=["8", 0], conditioning=["11", 0]),\n            "14": node("SamplerCustomAdvanced", noise=["12", 0], guider=["13", 0], sampler=["5", 0],\n                       sigmas=["9", 0], latent_image=["11", 1]),\n            "15": node("VAEDecode", samples=["14", 0], vae=["3", 0]),\n            "16": node("VAEDecodeAudio", samples=["14", 0], vae=["4", 0]),\n        })\n        images, audio = ["15", 0], ["16", 0]\n    else:\n        # Explicit shift agrees with Core\'s native 12/3 defaults; strict upstream\n        # compatibility checking stays enabled. No optional accelerator patches.\n        graph.update({\n            "6": node("MiniMaxH3SigmaShift", model=["1", 0], shift_video=12.0, shift_audio=3.0),\n            "7": node("BasicScheduler", model=["6", 0], scheduler="simple", steps=20, denoise=1.0),\n            "8": node("H3ContinuumSamplerV39", model=["6", 0], clip=["2", 0], video_vae=["3", 0],\n                      sampler=["5", 0], sigmas=["7", 0], sequence_prompt=prompt,\n                      prompt_mode="Fixed", chunks=chunks, chunk_seconds=float(duration_seconds),\n                      aspect="Landscape 16:9", preset="Draft — 0.30 MP", custom_mp=0.3,\n                      continuity="Balanced — 22 frames", base_seed=seed, audio_continuity=True,\n                      diagnostics="Detailed Report", reroll_from_chunk="Auto", reroll_nonce=0,\n                      strict_compatibility=True, debug=False, show_preview=False,\n                      run_storage="Save + Auto Resume", run_name=run_name,\n                      reference_size="Match Output", project_id=str(uuid.uuid5(uuid.NAMESPACE_URL, "colab-utils:" + run_name)),\n                      video_reference_size="Efficient - 0.4 MP", continuation_backend="Standard",\n                      generation_mode="Full Run", review_action="Continue / Next", take_group=0,\n                      take_revision_id="", take_action="Automatic", size_source="Manual", width=width, height=height),\n            "9": node("H3DecodeCacheHelper", cache_mode="Auto", ram_budget_mb=256, disk_budget_gb=8, reset_token=0,\n                      video_samples=["8", 0], audio_samples=["8", 1], video_vae=["3", 0], audio_vae=["4", 0]),\n            "10": node("H3ContinuumAssembleSeamV35", images=["9", 0], audio=["9", 1], assembly_plan=["8", 2],\n                       driving_audio=["8", 4], exact_total_duration=True, audio_seam="Auto", video_seam="Auto",\n                       buffer_backend="Auto", diagnostics="Detailed Report"),\n        })\n        images, audio = ["10", 0], ["10", 1]\n    graph["20"] = node("CreateVideo", images=images, audio=audio, fps=24.0)\n    graph["21"] = node("SaveVideo", video=["20", 0], filename_prefix=f"video/{preset}/{run_name}",\n                       **{"format": "mp4", "format.codec": "h264", "format.codec.encoding": "auto"})\n    return validate_attention(graph)\n\n\ndef export(directory, presets=PRESETS):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for preset in presets:\n        path = directory / (preset + ".api.json")\n        content = json.dumps(build_graph(preset), indent=2, ensure_ascii=False) + "\\n"\n        if path.exists() and path.read_text() != content:\n            raise FileExistsError(f"Starter has local edits; keep it and export to another directory: {path}")\n        path.write_text(content)\n        paths.append(path)\n    return paths\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    args = parser.parse_args()\n    for path in export(args.output_dir):\n        print(path)\n\n\nif __name__ == "__main__":\n    main()\n', 'fast_h3_client.py': '"""Small standard-library client for an already running ComfyUI server."""\n\nimport filecmp\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import quote, urlencode, urlsplit\nfrom urllib.request import Request, urlopen\nimport uuid\n\n\nclass GraphValidationError(ValueError):\n    """A graph cannot satisfy the server\'s advertised node schema."""\n\n\nclass ComfyHTTPError(RuntimeError):\n    def __init__(self, status, payload):\n        self.status, self.payload = status, payload\n        super().__init__(f"ComfyUI HTTP {status}: {payload}")\n\n\nclass PromptExecutionError(RuntimeError):\n    def __init__(self, prompt_id, history):\n        self.prompt_id, self.history = prompt_id, history\n        super().__init__(f"ComfyUI execution failed for prompt {prompt_id}")\n\n\nclass PromptTimeoutError(TimeoutError):\n    def __init__(self, prompt_id):\n        self.prompt_id = prompt_id\n        super().__init__(f"Timed out waiting for prompt {prompt_id}; it may still be running")\n\n\nclass ComfyClient:\n    def __init__(self, base_url="http://127.0.0.1:8188", timeout=30):\n        parsed = urlsplit(base_url)\n        if (parsed.scheme not in ("http", "https") or not parsed.hostname\n                or parsed.username or parsed.password or parsed.query or parsed.fragment):\n            raise ValueError("base_url must be an HTTP(S) server URL without credentials or query")\n        if timeout <= 0:\n            raise ValueError("timeout must be positive")\n        self.base_url, self.timeout = base_url.rstrip("/"), timeout\n\n    def _open(self, path, data=None, timeout=None):\n        if not path.startswith("/") or path.startswith("//"):\n            raise ValueError("API path must start with one slash")\n        body = None if data is None else json.dumps(data).encode("utf-8")\n        headers = {} if body is None else {"Content-Type": "application/json"}\n        request = Request(self.base_url + path, data=body, headers=headers)\n        try:\n            return urlopen(request, timeout=self.timeout if timeout is None else timeout)\n        except HTTPError as error:\n            with error:\n                raw = error.read().decode("utf-8", errors="replace")\n            try:\n                payload = json.loads(raw)\n            except ValueError:\n                payload = raw\n            raise ComfyHTTPError(error.code, payload) from error\n\n    def get_json(self, path):\n        with self._open(path) as response:\n            return json.load(response)\n\n    def post_json(self, path, data):\n        # A lost response does not imply the prompt was not queued. Never retry.\n        with self._open(path, data) as response:\n            return json.load(response)\n\n    def object_info(self):\n        return self.get_json("/object_info")\n\n    @staticmethod\n    def _input_specs(declared, inputs, prefix=""):\n        """Expand current ComfyUI DynamicCombo fields from their selected keys."""\n        specs, required = {}, set()\n        for group in ("required", "optional"):\n            for name, spec in declared.get(group, {}).items():\n                identity = prefix + name\n                specs[identity] = spec\n                if group == "required":\n                    required.add(identity)\n                if spec[0] == "COMFY_DYNAMICCOMBO_V3" and identity in inputs:\n                    options = spec[1].get("options", [])\n                    selected = next((option for option in options if option["key"] == inputs[identity]), None)\n                    if selected is None:\n                        raise GraphValidationError(f"Input {identity}: invalid dynamic combo selection")\n                    nested, needed = ComfyClient._input_specs(selected.get("inputs", {}), inputs, identity + ".")\n                    specs.update(nested)\n                    required.update(needed)\n        return specs, required\n\n    def validate_graph(self, graph):\n        """Check advertised inputs; /prompt still performs native validation."""\n        schemas = self.object_info()\n        if not isinstance(graph, dict) or not graph:\n            raise GraphValidationError("Graph must be a nonempty API node dictionary")\n        for node_id, node in graph.items():\n            if not isinstance(node_id, str) or not isinstance(node, dict):\n                raise GraphValidationError("API node IDs must be strings and nodes dictionaries")\n            class_type = node.get("class_type")\n            if not isinstance(class_type, str) or class_type not in schemas:\n                raise GraphValidationError(f"Node {node_id}: unknown class_type {class_type!r}")\n            inputs = node.get("inputs")\n            if not isinstance(inputs, dict):\n                raise GraphValidationError(f"Node {node_id}: inputs must be a dictionary")\n            declared = schemas[class_type].get("input", {})\n            specs, required = self._input_specs(declared, inputs)\n            missing = required - inputs.keys()\n            if missing:\n                raise GraphValidationError(f"Node {node_id}: missing inputs {sorted(missing)}")\n            for name, value in inputs.items():\n                label = f"Node {node_id} input {name}"\n                spec = specs.get(name)\n                if isinstance(value, list) and len(value) == 2 and isinstance(value[0], str):\n                    source, slot = value\n                    source_node = graph.get(source)\n                    source_class = source_node.get("class_type") if isinstance(source_node, dict) else None\n                    source_schema = schemas.get(source_class, {}) if isinstance(source_class, str) else {}\n                    outputs = source_schema.get("output", [])\n                    if source_node is None or type(slot) is not int or not 0 <= slot < len(outputs):\n                        raise GraphValidationError(f"{label}: invalid link {value!r}")\n                    continue\n                if not spec:\n                    continue  # Hidden/custom dynamic inputs are validated by ComfyUI.\n                kind = spec[0]\n                options = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}\n                if isinstance(kind, list):\n                    valid = value in kind\n                elif kind == "COMBO":\n                    choices = options.get("options", [])\n                    valid = (isinstance(value, list) and all(item in choices for item in value)\n                             if options.get("multiselect", False) else value in choices)\n                elif kind == "COMFY_DYNAMICCOMBO_V3":\n                    valid = any(option["key"] == value for option in options.get("options", []))\n                elif kind == "INT":\n                    valid = type(value) is int\n                elif kind == "FLOAT":\n                    valid = type(value) in (int, float) and math.isfinite(value)\n                elif kind == "BOOLEAN":\n                    valid = type(value) is bool\n                elif kind == "STRING":\n                    valid = isinstance(value, str)\n                else:\n                    valid = False  # Typed objects must arrive through links.\n                if not valid:\n                    raise GraphValidationError(f"{label}: invalid value for {kind!r}")\n                if kind in ("INT", "FLOAT") and any(\n                    key in options and compare(value, options[key])\n                    for key, compare in (("min", lambda a, b: a < b), ("max", lambda a, b: a > b))\n                ):\n                    raise GraphValidationError(f"{label}: value outside advertised range")\n        return graph\n\n    def queue_prompt(self, graph, prompt_id=None, extra_data=None):\n        self.validate_graph(graph)\n        identity = str(uuid.uuid4()) if prompt_id is None else str(prompt_id)\n        payload = {"prompt": graph, "prompt_id": identity}\n        if extra_data is not None:\n            payload["extra_data"] = extra_data\n        result = self.post_json("/prompt", payload)\n        if result.get("error") or result.get("node_errors") or not result.get("prompt_id"):\n            raise ComfyHTTPError(200, result)\n        return result\n\n    def _wait_json(self, path, prompt_id, deadline):\n        remaining = deadline - time.monotonic()\n        if remaining <= 0:\n            raise PromptTimeoutError(prompt_id)\n        try:\n            with self._open(path, timeout=min(self.timeout, remaining)) as response:\n                return json.load(response)\n        except TimeoutError as error:\n            raise PromptTimeoutError(prompt_id) from error\n        except URLError as error:\n            if isinstance(error.reason, TimeoutError):\n                raise PromptTimeoutError(prompt_id) from error\n            raise\n\n    def wait_prompt(self, prompt_id, timeout=1800, poll_seconds=5, on_status=None):\n        if timeout < 0 or poll_seconds <= 0:\n            raise ValueError("timeout must be nonnegative and poll_seconds positive")\n        started = time.monotonic()\n        deadline = started + timeout\n        while True:\n            history = self._wait_json("/history/" + quote(str(prompt_id), safe=""),\n                                      prompt_id, deadline).get(str(prompt_id))\n            status = history.get("status", {}) if history else {}\n            messages = status.get("messages", [])\n            failed = status.get("status_str") in ("error", "failed", "interrupted") or any(\n                message and message[0] in ("execution_error", "execution_interrupted", "execution_failed")\n                for message in messages\n            )\n            if failed:\n                raise PromptExecutionError(prompt_id, history)\n            complete = status.get("completed") is True and status.get("status_str") == "success"\n            state = "completed" if complete else "unknown"\n            if not complete:\n                queue = self._wait_json("/queue", prompt_id, deadline)\n                for key, value in (("queue_running", "running"), ("queue_pending", "pending")):\n                    if any(len(entry) > 1 and str(entry[1]) == str(prompt_id) for entry in queue.get(key, [])):\n                        state = value\n                        break\n            if on_status:\n                on_status({"prompt_id": str(prompt_id), "state": state,\n                           "elapsed_seconds": time.monotonic() - started, "status": status})\n            if complete:\n                return history\n            remaining = deadline - time.monotonic()\n            if remaining <= 0:\n                raise PromptTimeoutError(prompt_id)\n            time.sleep(min(poll_seconds, remaining))\n\n    @staticmethod\n    def output_files(history):\n        files = []\n        for output in history.get("outputs", {}).values():\n            for key in ("videos", "gifs", "images"):\n                for item in output.get(key, []):\n                    filename = item.get("filename", "")\n                    if Path(filename).suffix.lower() in (".mp4", ".webm", ".mov"):\n                        entry = {"filename": filename, "subfolder": item.get("subfolder", ""),\n                                 "type": item.get("type", "output")}\n                        if entry not in files:\n                            files.append(entry)\n        return files\n\n    def download_output(self, item, path):\n        target = Path(path)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        query = urlencode({key: item.get(key, default) for key, default in\n                           (("filename", ""), ("subfolder", ""), ("type", "output"))})\n        fd, temporary = tempfile.mkstemp(prefix="." + target.name + ".", dir=target.parent)\n        try:\n            with os.fdopen(fd, "wb") as output, self._open("/view?" + query) as response:\n                shutil.copyfileobj(response, output)\n                output.flush()\n                os.fsync(output.fileno())\n            # Hard-link publication is atomic and cannot replace an existing file.\n            # Unlike os.replace(), it also preserves a target created concurrently.\n            try:\n                os.link(temporary, target)\n            except FileExistsError:\n                if not filecmp.cmp(temporary, target, shallow=False):\n                    raise FileExistsError(f"Output already exists with different bytes: {target}")\n            return target\n        finally:\n            os.unlink(temporary)\n', 'comfyui_catalog.json': '{\n  "schema_version": 1,\n  "checked_date": "2026-10-28",\n  "comfyui": {\n    "repo": "https://github.com/Comfy-Org/ComfyUI.git",\n    "revision": "8f04fd6ecf4a548937971e840af2a985529fab76",\n    "version": "0.42.0"\n  },\n  "extensions": {\n    "continuum": {\n      "repo": "https://github.com/ukr8b3g-cmyk/ComfyUI-H3-Continuum.git",\n      "revision": "7e14fde33f8e8fa5804fa409777d33afee2427f4",\n      "directory": "ComfyUI-H3-Continuum",\n      "expected_nodes": [\n        "H3ContinuumSamplerV39",\n        "H3DecodeCacheHelper",\n        "H3ContinuumAssembleSeamV35"\n      ],\n      "requirements": [],\n      "license": "MIT"\n    },\n    "ref2va-vsa": {\n      "repo": "https://github.com/Kablex/ComfyUI-Ref2VA-VSA.git",\n      "revision": "323a0a5f7c636cc70f95464145732f81f04a72e9",\n      "directory": "ComfyUI-Ref2VA-VSA",\n      "expected_nodes": [\n        "Ref2VAVSAGatePatch",\n        "FastH3VSAPatch"\n      ],\n      "requirements": [\n        "torch>=2.4.0",\n        "safetensors>=0.4.3"\n      ],\n      "license": "Apache-2.0"\n    }\n  },\n  "assets": {\n    "h3-omni-int8": {\n      "repo": "FastVideo/FastH3-Omni-Comfy",\n      "revision": "82eff2a6eee6c962846879f4bab52610a4df24d3",\n      "file": "diffusion_models/fasth3_omniref_8step_pruned_int8_convrot.safetensors",\n      "size": 22213723064,\n      "sha256": "851b2dc9670ed864b7f1e951ac9da047d83b1c04ded9bfcc1a9e5bba0ee33f80",\n      "destination": "diffusion_models/fasth3_omniref_8step_pruned_int8_convrot.safetensors"\n    },\n    "h3-text-int8": {\n      "repo": "FastVideo/FastH3-Omni-Comfy",\n      "revision": "82eff2a6eee6c962846879f4bab52610a4df24d3",\n      "file": "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",\n      "size": 27141342152,\n      "sha256": "bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923",\n      "destination": "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"\n    },\n    "h3-fl2va-int8": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",\n      "size": 20970379616,\n      "sha256": "e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a",\n      "destination": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors"\n    },\n    "h3-ref2va-int8": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",\n      "size": 20970379616,\n      "sha256": "9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779",\n      "destination": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"\n    },\n    "h3-text-awq": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n      "size": 15687142551,\n      "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6",\n      "destination": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"\n    },\n    "h3-video-int8": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "vae/minimax_h3_video_vae_int8_convrot.safetensors",\n      "size": 2811065184,\n      "sha256": "52a2c8c73583c86e4f41cdcce3a6ad0ea562987bc0bf3d60a0cef5f5c8e60c0e",\n      "destination": "vae/minimax_h3_video_vae_int8_convrot.safetensors"\n    },\n    "h3-video-fp16": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "vae/minimax_h3_video_vae_fp16.safetensors",\n      "size": 5207808496,\n      "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522",\n      "destination": "vae/minimax_h3_video_vae_fp16.safetensors"\n    },\n    "h3-audio-fp32": {\n      "repo": "Comfy-Org/MiniMax-H3",\n      "revision": "e5eb578a89295337b8ff433a035929ce0279e0b6",\n      "file": "vae/minimax_h3_audio_vae_fp32.safetensors",\n      "size": 605254808,\n      "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48",\n      "destination": "vae/minimax_h3_audio_vae_fp32.safetensors"\n    },\n    "h3-ref-turbo4": {\n      "repo": "lightx2v/Minimax-h3-Turbo",\n      "revision": "a29e57d195f80ead07c89f89e55ecf2911e893c5",\n      "file": "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",\n      "size": 1956193000,\n      "sha256": "5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c",\n      "destination": "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"\n    },\n    "h3-ref-vsa-gate": {\n      "repo": "barelymining/ComfyUI-MiniMax-H3-FastVideo",\n      "revision": "aa554050fe27ec3c62e20e57519beb46741c9000",\n      "file": "fasth3_vsa_gate.safetensors",\n      "size": 3853522616,\n      "sha256": "bf95408e335c4b0d6a1a44946a428a3e12849e6eab2de9566ecedae4a0043420",\n      "destination": "loras/fasth3_vsa_gate.safetensors"\n    }\n  },\n  "bundles": {\n    "none": {\n      "extensions": [],\n      "assets": [],\n      "description": "Core/custom nodes only; no model weights downloaded."\n    },\n    "h3-omni-int8": {\n      "extensions": [],\n      "assets": [\n        "h3-omni-int8",\n        "h3-text-int8",\n        "h3-video-int8",\n        "h3-audio-fp32"\n      ],\n      "description": "FastVideo FastH3 Omni Ref2VA INT8, vision-capable INT8 Qwen3-VL, INT8 video VAE and FP32 audio VAE. Native dense attention, 8-step Euler/simple, guidance 1, shifts 12/3. No extra LoRA or gate transplant. Default reference sizing: max. GPU generation not yet tested."\n    },\n    "h3-continuum": {\n      "extensions": [\n        "continuum"\n      ],\n      "assets": [\n        "h3-fl2va-int8",\n        "h3-text-awq",\n        "h3-video-int8",\n        "h3-audio-fp32"\n      ],\n      "description": "Native FL2VA INT8, core attention, 20-step res_multistep, two 5-second chunks. No speed LoRA."\n    },\n    "h3-ref2va-vsa": {\n      "extensions": [\n        "ref2va-vsa"\n      ],\n      "assets": [\n        "h3-ref2va-int8",\n        "h3-text-awq",\n        "h3-video-fp16",\n        "h3-audio-fp32",\n        "h3-ref-turbo4",\n        "h3-ref-vsa-gate"\n      ],\n      "description": "EXPERIMENTAL native Ref2VA + gate transplant, task-matched Turbo4, Euler/simple and 75% sparsity. NVIDIA 24 GB class with native CUDA Sol kernel required."\n    }\n  },\n  "deferred": [\n    "Veda: researched only; user explicitly deferred implementation."\n  ]\n}\n'}
# END EMBEDDED COMFYUI WORKSPACE TOOLKIT

toolkit = Path(TOOLKIT_ROOT)
toolkit.mkdir(parents=True, exist_ok=True)
for filename, source in COMFYUI_SOURCES.items():
    destination = toolkit / filename
    if not destination.is_file() or destination.read_text() != source:
        destination.write_text(source)
if str(toolkit) not in sys.path:
    sys.path.insert(0, str(toolkit))
from comfyui_workspace import ComfyWorkspace
extensions = [name for name, enabled in (
    ("continuum", ENABLE_CONTINUUM), ("ref2va-vsa", ENABLE_REF2VA_VSA)) if enabled]
workspace = ComfyWorkspace(root=WORKSPACE_ROOT, port=COMFYUI_PORT, extensions=extensions)
print("Workspace:", workspace.root, "Port:", COMFYUI_PORT, "Extensions:", extensions)


In [ ]:
# @title 2. 離線預覽版本與模型磁碟下載量（GiB；不是 VRAM）
WORKSPACE_PLAN = workspace.plan(bundle=MODEL_BUNDLE)
print(json.dumps(WORKSPACE_PLAN, indent=2, ensure_ascii=False, default=str))


In [ ]:
# @title 3. 安裝並準備 API starter（可關閉 INSTALL_COMFYUI）
if INSTALL_COMFYUI:
    print(json.dumps(workspace.install(), indent=2, ensure_ascii=False, default=str))
    print(json.dumps(workspace.stage_workflows(), indent=2, ensure_ascii=False, default=str))
else:
    print("略過安裝；先檢視第 2 格計畫，再將 INSTALL_COMFYUI 設為 True 並重跑設定。")


In [ ]:
# @title 4. 可選模型下載（先查看第 2 格 GiB）
if MODEL_BUNDLE != "none":
    print(json.dumps(workspace.download_models(bundle=MODEL_BUNDLE), indent=2, ensure_ascii=False, default=str))
else:
    print("略過模型下載；明確選擇 MODEL_BUNDLE 才下載。")


In [ ]:
# @title 5. 啟動 ComfyUI 與 Colab 原生 port iframe
if START_COMFYUI:
    print(json.dumps(workspace.start(timeout=180), indent=2, ensure_ascii=False, default=str))
    if SHOW_COMFYUI:
        try:
            from google.colab import output
        except ImportError:
            print(f"非 Colab 環境：請開啟 http://127.0.0.1:{COMFYUI_PORT}")
        else:
            output.serve_kernel_port_as_iframe(COMFYUI_PORT, height=900)
else:
    print("略過啟動；安裝完成後，將 START_COMFYUI 設為 True 並重跑設定。")


## B. 可選 API 執行（預設不生成）

可在 UI 建立工作流並匯出 **API 格式 JSON**，放入 `API_WORKFLOW_PATH`；或選擇下方 starter。安裝格會把 starter 放在 `<root>/workflows/`。API 執行是此工具驗證的接線方式；未驗證這些 API JSON 能直接拖入 UI 編輯器。

圖片先上傳至 `<root>/ComfyUI/input/`，`REFERENCE_IMAGE` 填相對檔名。Ref2VA 需使用原生 Ref2VA 的六欄提示與對應 reference 編號。Continuum starter 使用 Core attention，沒有上游範本額外 KJ、Spectrum、rgthree 依賴。Continuum starter 使用原生 20-step res_multistep，未加 Turbo；Ref2VA-VSA 使用對應 Ref2VA Turbo4 / Euler。非 Continuum 的原生 H3 影格需對齊 17n+5，因此輸入 5 秒可產生 124 幀（約 5.167 秒），不保證恰為 5 秒。PROMPT 留空會使用所選 preset 對應的提示範例；自填 Ref2VA 提示必須有六欄及 `<Picture 1>`。先下載所選 preset 的模型，再開啟 RUN_API。server 啟動後更換模型 bundle，需先停止 ComfyUI、下載，再啟動以更新 loader 清單。工作流或擴充改動後需重新確認相容性。

Omni starter 使用原生 Ref2VA、8-step Euler/simple 與 dense attention。`REF_IMAGE_SIZE` 適用 Ref2VA-VSA 與 Omni：`max` 保留最高 2048px 短邊的參考圖細節（不放大小圖）；`match` 按輸出像素面積縮小，較快省記憶體。表單預設 `max`，可能慢數倍、增加記憶體，不能保證身份一致；這不是 FL2VA + Ref2VA hybrid 權重。此 starter 接一張圖片；多圖／影片／音訊 references 可使用自訂 API graph。


In [ ]:
# @title 6. 可選 API 工作設定
RUN_API = False # @param {type:"boolean"}
API_WORKFLOW_PATH = "" # @param {type:"string"}
PRESET = "h3-continuum" # @param ["h3-continuum", "h3-ref2va-vsa", "h3-omni-int8"]
PROMPT = "" # @param {type:"string"}
# Empty uses the selected preset task-matched example; Ref2VA custom text needs all six fields and <Picture 1>.
WIDTH = 864 # @param {type:"integer"}
HEIGHT = 480 # @param {type:"integer"}
DURATION_SECONDS = 5 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
REFERENCE_IMAGE = "reference.png" # @param {type:"string"}
REF_IMAGE_SIZE = "max" # @param ["max", "match"]
CHUNKS = 2 # @param {type:"integer"}
RUN_NAME = "comfy_h3" # @param {type:"string"}
API_TIMEOUT = 1800 # @param {type:"integer"}


In [ ]:
# @title 7. 明確開啟 RUN_API 後送出 API 工作流
if RUN_API:
    workflow_path = Path(API_WORKFLOW_PATH) if API_WORKFLOW_PATH else workspace.root / "workflows" / "notebook_job.json"
    if not API_WORKFLOW_PATH:
        from comfyui_workflows import build_graph
        graph = build_graph(PRESET, prompt=PROMPT, width=WIDTH, height=HEIGHT,
                            duration_seconds=DURATION_SECONDS, seed=SEED,
                            reference_image=REFERENCE_IMAGE, chunks=CHUNKS, run_name=RUN_NAME,
                            ref_image_size=REF_IMAGE_SIZE)
        workflow_path.parent.mkdir(parents=True, exist_ok=True)
        workflow_path.write_text(json.dumps(graph, indent=2, ensure_ascii=False) + "\n")
    API_RESULT = workspace.run_api(workflow_path, timeout=API_TIMEOUT)
    print(json.dumps(API_RESULT, indent=2, ensure_ascii=False, default=str))
else:
    print("RUN_API=False：未建立或執行生成工作。")


## 狀態、下載與關閉

結果通常在 `<root>/ComfyUI/output/`；API 回傳資訊會列出輸出。先播放影片、檢查音訊，再下載需要的檔案。在 Colab 左側 Files 選檔下載，或另開一格執行 `from google.colab import files; files.download('/content/comfyui-workspace/ComfyUI/output/你的檔名.mp4')`。

Continuum 的 Take/session 資料需與輸出一起保存；依所選 `run_storage` 確認實際路徑，離開前備份，不要只下載最終 MP4。`/content` 會隨 runtime 消失。下方只停止此工具管理的 ComfyUI 程序；**另從 Colab 選單停止／刪除 runtime 才能釋放配額**。


In [ ]:
# @title 8. 查看狀態（不啟動或下載）
print(json.dumps(workspace.status(), indent=2, ensure_ascii=False, default=str))


In [ ]:
# @title 9. 可選：只停止 ComfyUI 程序（不停止 Colab runtime）
STOP_COMFYUI = False # @param {type:"boolean"}
if STOP_COMFYUI:
    print(json.dumps(workspace.stop(), indent=2, ensure_ascii=False, default=str))
else:
    print("STOP_COMFYUI=False：保留 server。完成下載後請另行停止 Colab runtime。")
